# Fine-tune FLAN-T5-Small with LoRA for Empathetic Chat

## 1. Import Libraries

In [24]:
import sys
import subprocess

# Install packages using the notebook's Python interpreter
packages = [
    'pandas', 'numpy', 'transformers==4.40.0', 'datasets==2.19.0',
    'peft==0.10.0', 'accelerate==0.29.0', 'evaluate==0.4.1',
    'rouge_score', 'nltk', 'sentencepiece', 'protobuf', 'tqdm'
]

for package in packages:
    print(f"Installing {package}...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print("\n✓ All packages installed!")

Installing pandas...
Installing numpy...
Installing transformers==4.40.0...
Installing datasets==2.19.0...
Installing peft==0.10.0...
Installing accelerate==0.29.0...
Installing evaluate==0.4.1...
Installing rouge_score...
Installing nltk...
Installing sentencepiece...
Installing protobuf...
Installing tqdm...

✓ All packages installed!


In [25]:
import pandas as pd
import transformers
print("✓ Packages imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Transformers version: {transformers.__version__}")

✓ Packages imported successfully!
Pandas version: 3.0.0
Transformers version: 4.40.0


In [26]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import numpy as np
import nltk

# Check GPU availability
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected. Training will be slow!")

# Download NLTK data for ROUGE
nltk.download('punkt', quiet=True)
print("\n✓ Libraries imported successfully!")

PyTorch version: 2.7.1+cu118
CUDA available: True
GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU
GPU Memory: 8.59 GB

✓ Libraries imported successfully!


## 2. Load Dataset

**NOTE:** Update the dataset path to match your local data location

In [27]:
import pandas as pd
from datasets import Dataset

# Correct path and file format
DATA_PATH = "clean_dataset_3"

# Load JSONL files (note the .jsonl extension and lines=True parameter)
train_df = pd.read_json(f"{DATA_PATH}/train_en.jsonl", lines=True)
val_df = pd.read_json(f"{DATA_PATH}/val_en.jsonl", lines=True)

dataset = {
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df)
}

print(f"Training samples: {len(dataset['train'])}")
print(f"Validation samples: {len(dataset['validation'])}")
print("\n✓ Dataset loaded!")

Training samples: 799
Validation samples: 100

✓ Dataset loaded!


## 3. Load Model and Tokenizer

In [28]:
MODEL_NAME = "google/flan-t5-small"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.float32  # Changed from torch.float16 to torch.float32
)

print(f"Model loaded on: {model.device}")
print("\n✓ Model and tokenizer ready!")

Loading google/flan-t5-small...


C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded on: cuda:0

✓ Model and tokenizer ready!


## 4. Configure LoRA

In [29]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=64,                    # LoRA rank
    lora_alpha=128,          # LoRA alpha
    lora_dropout=0.1,       # Dropout
    target_modules=["q","k","v","o"],  # Apply to query and value matrices
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("\n✓ LoRA configured!")

trainable params: 5,505,024 || all params: 82,466,176 || trainable%: 6.675493234947623

✓ LoRA configured!


## 5. Tokenize Dataset

In [30]:
MAX_INPUT_LENGTH = 256
MAX_TARGET_LENGTH = 128

def preprocess_function(examples):
    """Tokenize prompts and targets"""
    inputs = examples["prompt"]
    targets = examples["target"]
    
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    labels = tokenizer(
        targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset...")
tokenized_dataset = {
    "train": dataset["train"].map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names),
    "validation": dataset["validation"].map(preprocess_function, batched=True, remove_columns=dataset["validation"].column_names)
}

print("✓ Dataset tokenized!")

Tokenizing dataset...


Map: 100%|██████████| 100/100 [00:00<00:00, 6054.83 examples/s]

✓ Dataset tokenized!


## 6. Setup Evaluation Metrics

In [31]:
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    """Calculate ROUGE scores"""
    predictions, labels = eval_pred
    
    # Replace -100 in predictions (which are used for padding)
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)

    # Decode predictions and labels
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Compute ROUGE
    result = rouge_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    
    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"],
        "rougeLsum": result["rougeLsum"],
    }

print("✓ Metrics configured!")

✓ Metrics configured!


## 7. Training Configuration

In [33]:
OUTPUT_DIR = "./flan-t5-cbt-799-40epochs-v1"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training hyperparameters
    num_train_epochs=40,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    warmup_steps=500,
    weight_decay=0.01,

    # Optimization
    fp16=False,  #
    gradient_accumulation_steps=8,

    # Evaluation
    evaluation_strategy="steps",
    save_strategy="steps",
    eval_steps=20,
    save_steps=20,
    save_total_limit=5,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",

    # Logging
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
    
    # Generation settings for evaluation
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Mixed precision (FP16): {training_args.fp16}")
print(f"Batch size per device: {training_args.per_device_train_batch_size}")
print("\n✓ Training arguments configured!")

Output directory: ./flan-t5-cbt-799-40epochs-v1
Mixed precision (FP16): False
Batch size per device: 2

✓ Training arguments configured!


## 8. Initialize Trainer

In [34]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("✓ Trainer initialized!")

✓ Trainer initialized!


## 9. Train the Model

In [35]:
print("="*60)
print("STARTING TRAINING")
print("="*60)

trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)

STARTING TRAINING


Step,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
20,37.289000,39.436081,0.062271,0.004408,0.057200,0.057366
40,37.123900,39.235268,0.062271,0.004408,0.057200,0.057366
60,36.644300,38.888443,0.063042,0.004408,0.058077,0.058194
80,36.419900,38.368771,0.066404,0.006337,0.061332,0.061290
100,35.456800,37.644173,0.069348,0.006337,0.064056,0.064236
120,34.926900,36.665787,0.066553,0.004381,0.061445,0.061547
140,33.976500,35.374702,0.063631,0.004402,0.059495,0.059656
160,32.419400,33.695061,0.062692,0.004402,0.058759,0.058649
180,31.236700,31.509495,0.059076,0.003409,0.054461,0.054780
200,29.130800,28.757658,0.056236,0.002400,0.052913,0.052992


C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.ven


TRAINING COMPLETED


## 10. Evaluate on Validation Set

In [36]:
print("\nEvaluating on validation set...")
eval_results = trainer.evaluate()

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
for key, value in eval_results.items():
    print(f"  {key:30s}: {value:.4f}")
print("="*60)


Evaluating on validation set...



EVALUATION RESULTS
  eval_loss                     : 2.3003
  eval_rouge1                   : 0.1247
  eval_rouge2                   : 0.0298
  eval_rougeL                   : 0.1130
  eval_rougeLsum                : 0.1127
  eval_runtime                  : 24.2892
  eval_samples_per_second       : 4.1170
  eval_steps_per_second         : 0.5350
  epoch                         : 40.0000


## 11. Save the Model

In [37]:
# Save the LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✓ Model saved to {OUTPUT_DIR}")
print("\nTo load the model later:")
print(f"  from peft import AutoPeftModelForSeq2SeqLM")
print(f"  model = AutoPeftModelForSeq2SeqLM.from_pretrained('{OUTPUT_DIR}')")

C:\Users\PAVISHANTH\Desktop\DSGP\Implementation\.venv2\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✓ Model saved to ./flan-t5-cbt-799-40epochs-v1

To load the model later:
  from peft import AutoPeftModelForSeq2SeqLM
  model = AutoPeftModelForSeq2SeqLM.from_pretrained('./flan-t5-cbt-799-40epochs-v1')


## 12. Generation Function

In [38]:
def generate_response(prompt, **kwargs):
    """
    Generate a clean chatbot response by stripping prompt headers.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Extract params with defaults
    max_new_tokens = kwargs.get("max_length", 150)
    temperature = kwargs.get("temperature", 0.7)
    top_p = kwargs.get("top_p", 0.9)
    do_sample = kwargs.get("do_sample", True)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Decode the full output
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Clean the output
    if "Assistant:" in full_text:
        response = full_text.split("Assistant:")[-1].strip()
    else:
        response = full_text.strip()
        
    return response

print("✓ Generation function ready!")

✓ Generation function ready!


## 13. Test with Sample Examples

In [39]:
print("\n" + "="*60)
print("TESTING MODEL GENERATION")
print("="*60 + "\n")

# Test with validation examples
num_samples = min(3, len(dataset["validation"]))

for i in range(num_samples):
    sample = dataset["validation"][i]
    prompt = sample["prompt"]
    target = sample["target"]
    
    print(f"Example {i+1}:")
    print("-" * 60)
    print(f"Prompt:\n{prompt}\n")
    print(f"Expected Response:\n{target}\n")
    
    generated = generate_response(prompt)
    print(f"Generated Response:\n{generated}\n")
    print("=" * 60 + "\n")


TESTING MODEL GENERATION

Example 1:
------------------------------------------------------------
Prompt:
Context: The user is feeling angry. Task: Respond as an empathetic assistant.
User: I can’t tolerate people interrupting me.
Assistant:

Expected Response:
Interruptions are frustrating. Responding calmly or choosing when to engage helps maintain control over your emotions.

Generated Response:
Accepting silence is a way to gain control.


Example 2:
------------------------------------------------------------
Prompt:
Context: The user is feeling disgust. Task: Respond as an empathetic assistant.
User: I feel disgusted by my lack of motivation.
Assistant:

Expected Response:
That’s a tough feeling, but lack of motivation often stems from burnout or fear. Let’s explore what’s draining your energy.

Generated Response:
Self-disappearance is a key attribute of your ability.


Example 3:
------------------------------------------------------------
Prompt:
Context: The user is feeling 

## 14. Interactive Testing

In [40]:
# ============================================================================
# INTERACTIVE TESTING - CHATBOT MODE
# ============================================================================

# 1. SET YOUR INPUTS
user_input = "I'm feeling really happy today because I got promoted at work!"
user_emotion = "joy"  # Options: joy, sad, angry, disgust, etc.

# 2. GENERATION SETTINGS
generation_params = {
    "max_length": 150,
    "temperature": 0.7,
    "top_p": 0.9,
    "do_sample": True,
}

# 3. CONSTRUCT THE CHAT PROMPT
chat_prompt = (
    f"Context: The user is feeling {user_emotion}. "
    f"Task: Respond as an empathetic assistant.\n"
    f"User: {user_input}\n"
    f"Assistant:"
)

print("="*60)
print("INTERACTIVE MODEL TEST")
print("="*60)
print(f"User Mood: {user_emotion}")
print(f"User Says: {user_input}")
print("-" * 60)

# Generate response
response = generate_response(chat_prompt, **generation_params)

print(f"\nAssistant: {response}")
print("\n" + "="*60)

INTERACTIVE MODEL TEST
User Mood: joy
User Says: I'm feeling really happy today because I got promoted at work!
------------------------------------------------------------

Assistant: Happy people are the way to go.



## 15. Batch Testing

In [42]:
# ============================================================================
# BATCH TESTING - Add multiple test inputs
# ============================================================================

test_inputs = [
    "I just lost my job and I don't know what to do.",
    "I'm so excited! I'm going on vacation next week!",
    "My best friend betrayed my trust.",
    "I can't believe I finally finished my degree!",
    "I'm feeling overwhelmed with all my responsibilities.",
]

print("="*60)
print("BATCH TESTING")
print("="*60 + "\n")

for idx, user_input in enumerate(test_inputs, 1):
    print(f"Test {idx}/{len(test_inputs)}")
    print("-" * 60)
    print(f"Input: {user_input}")
    
    response = generate_response(
        user_input,
        max_length=150,
        temperature=0.8,
        top_p=0.9,
        do_sample=True
    )
    
    print(f"Response: {response}")
    print("\n")

BATCH TESTING

Test 1/5
------------------------------------------------------------
Input: I just lost my job and I don't know what to do.
Response: I want to talk about what you do for life.


Test 2/5
------------------------------------------------------------
Input: I'm so excited! I'm going on vacation next week!
Response: I want to take a vacation with you to see what the trip is like.


Test 3/5
------------------------------------------------------------
Input: My best friend betrayed my trust.
Response: This trust was a reminder of how close you were to it.


Test 4/5
------------------------------------------------------------
Input: I can't believe I finally finished my degree!
Response: You've never mastered it before.


Test 5/5
------------------------------------------------------------
Input: I'm feeling overwhelmed with all my responsibilities.
Response: This is a difficult situation.




In [22]:
# Check a few examples from your training data
print("Sample training examples:")
print("="*60)
for i in range(3):
    sample = dataset["train"][i]
    print(f"\nExample {i+1}:")
    print(f"Prompt: {sample['prompt'][:200]}...")  # First 200 chars
    print(f"Target: {sample['target'][:200]}...")
    print("-"*60)

Sample training examples:

Example 1:
Prompt: Context: The user is feeling sad. Task: Respond as an empathetic assistant.
User: I regret trusting people.
Assistant:...
Target: That regret shows hurt. Could we examine what specifically led to that pain and what you learned about setting healthy boundaries?...
------------------------------------------------------------

Example 2:
Prompt: Context: The user is feeling surprised. Task: Respond as an empathetic assistant.
User: I can’t believe how peaceful I feel after journaling.
Assistant:...
Target: That’s wonderful to hear. Writing can help you sort out your thoughts and release tension, which naturally brings more calm....
------------------------------------------------------------

Example 3:
Prompt: Context: The user is feeling fearful. Task: Respond as an empathetic assistant.
User: I feel nervous during presentations.
Assistant:...
Target: Presentation anxiety is common. Focusing on preparation and breathing deeply helps maintain